In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import os
import shutil
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

DATASET_ROOT   = '/content/drive/MyDrive/Data'
SYNTHETIC_ROOT = '/content/drive/MyDrive/xray_synthetic'
COMBINED_ROOT  = '/content/drive/MyDrive/xray_combined'
CLASSES        = ['organic', 'inorganic']
NUM_CLASSES    = len(CLASSES)

IMAGE_SIZE  = 64
BATCH_SIZE  = 64
LATENT_DIM  = 100
EPOCHS_GAN  = 200
LR_G = LR_D = 0.0002

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')

# ====================== STEP 1: BUILD CLEAN DATASET (images only) ======================
CLEAN_ROOT = '/content/drive/MyDrive/xray_clean'
print("🧹 Building clean dataset (organic + inorganic only)...")

for cls in CLASSES:
    src = os.path.join(DATASET_ROOT, cls)
    dst = os.path.join(CLEAN_ROOT, cls)
    os.makedirs(dst, exist_ok=True)
    count = 0
    for f in os.listdir(src):
        if f.lower().endswith(IMAGE_EXTS):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
            count += 1
    print(f"   {cls}: {count} images")

print("✅ Clean dataset ready\n")

# ====================== DCGAN MODELS ======================
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x): return self.main(x)

class Discriminator(nn.Module):
    def __init__(self, ndf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x): return self.main(x)

def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# ====================== TRAIN GAN PER CLASS ======================
def train_gan_for_class(class_name, class_idx):
    print(f"\n🔥 Training GAN for {class_name}...")
    full_dataset  = datasets.ImageFolder(root=CLEAN_ROOT, transform=transform)
    class_indices = [i for i, t in enumerate(full_dataset.targets) if t == class_idx]

    if len(class_indices) == 0:
        raise RuntimeError(f"No images found for '{class_name}'.")

    print(f"   {len(class_indices)} images found")
    dataloader = DataLoader(Subset(full_dataset, class_indices),
                            batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

    netG = Generator().to(device);     netG.apply(weights_init)
    netD = Discriminator().to(device); netD.apply(weights_init)
    criterion  = nn.BCELoss()
    optimizerD = optim.Adam(netD.parameters(), lr=LR_D, betas=(0.5, 0.999))
    optimizerG = optim.Adam(netG.parameters(), lr=LR_G, betas=(0.5, 0.999))

    for epoch in range(EPOCHS_GAN):
        for i, (real_imgs, _) in enumerate(dataloader):
            real_imgs  = real_imgs.to(device)
            b_size     = real_imgs.size(0)
            label_real = torch.ones(b_size,  device=device)
            label_fake = torch.zeros(b_size, device=device)

            netD.zero_grad()
            loss_D_real = criterion(netD(real_imgs).view(-1), label_real)
            loss_D_real.backward()
            noise     = torch.randn(b_size, LATENT_DIM, 1, 1, device=device)
            fake_imgs = netG(noise)
            loss_D_fake = criterion(netD(fake_imgs.detach()).view(-1), label_fake)
            loss_D_fake.backward()
            optimizerD.step()

            netG.zero_grad()
            loss_G = criterion(netD(fake_imgs).view(-1), label_real)
            loss_G.backward()
            optimizerG.step()

            if i % 50 == 0:
                print(f"  Epoch [{epoch+1}/{EPOCHS_GAN}] "
                      f"Loss_D: {loss_D_real.item()+loss_D_fake.item():.4f}  "
                      f"Loss_G: {loss_G.item():.4f}")

    os.makedirs(SYNTHETIC_ROOT, exist_ok=True)
    torch.save(netG.state_dict(), f"{SYNTHETIC_ROOT}/generator_{class_name}.pth")
    print(f"✅ GAN for {class_name} done!")
    return netG

def generate_synthetic(class_name, netG, num_images=400):
    save_dir = os.path.join(SYNTHETIC_ROOT, class_name)
    os.makedirs(save_dir, exist_ok=True)
    netG.eval()
    with torch.no_grad():
        for i in range(num_images):
            noise = torch.randn(1, LATENT_DIM, 1, 1, device=device)
            fake  = (netG(noise).cpu().squeeze(0) * 0.5 + 0.5).clamp(0, 1)
            transforms.ToPILImage()(fake).save(os.path.join(save_dir, f"syn_{i:04d}.png"))
    print(f"✅ Generated {num_images} synthetic {class_name} images")

# ====================== TRAIN GANS ======================
for idx, cls in enumerate(CLASSES):
    netG = train_gan_for_class(cls, idx)
    generate_synthetic(cls, netG, num_images=400)

print("\n🎉 All GANs trained!")

# ====================== BUILD COMBINED DATASET ======================
print("\n🔗 Building combined dataset...")
if os.path.exists(COMBINED_ROOT):
    shutil.rmtree(COMBINED_ROOT)

for cls in CLASSES:
    dst = os.path.join(COMBINED_ROOT, cls)
    os.makedirs(dst, exist_ok=True)

    # Real images
    for f in os.listdir(os.path.join(CLEAN_ROOT, cls)):
        shutil.copy2(os.path.join(CLEAN_ROOT, cls, f), os.path.join(dst, f))

    # Synthetic images
    for f in os.listdir(os.path.join(SYNTHETIC_ROOT, cls)):
        shutil.copy(os.path.join(SYNTHETIC_ROOT, cls, f), os.path.join(dst, f))

    count = len(os.listdir(dst))
    print(f"   {cls}: {count} total images (real + synthetic)")

print("✅ Combined dataset ready")

# ====================== FINAL CLASSIFIER ======================
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

combined_dataset = datasets.ImageFolder(root=COMBINED_ROOT, transform=transform)
train_size = int(0.8 * len(combined_dataset))
val_size   = len(combined_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(combined_dataset, [train_size, val_size])
train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=0)

print("\n🚀 Training final classifier...")
for epoch in range(15):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}/15 — Loss: {running_loss/len(train_loader):.4f}")

model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(model(images), 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n✅ Final Accuracy: {100 * correct / total:.2f}%")
torch.save(model.state_dict(), '/content/drive/MyDrive/xray_classifier_2class.pth')
print("💾 Model saved!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
🧹 Building clean dataset (organic + inorganic only)...
   organic: 451 images
   inorganic: 451 images
✅ Clean dataset ready


🔥 Training GAN for organic...
   451 images found
  Epoch [1/200] Loss_D: 1.5402  Loss_G: 4.7188
  Epoch [2/200] Loss_D: 0.3146  Loss_G: 7.5708
  Epoch [3/200] Loss_D: 0.2173  Loss_G: 6.2360
  Epoch [4/200] Loss_D: 0.0045  Loss_G: 11.6849
  Epoch [5/200] Loss_D: 0.6650  Loss_G: 24.6933
  Epoch [6/200] Loss_D: 4.8276  Loss_G: 24.5707
  Epoch [7/200] Loss_D: 1.6574  Loss_G: 26.0807
  Epoch [8/200] Loss_D: 0.0141  Loss_G: 10.3837
  Epoch [9/200] Loss_D: 0.2636  Loss_G: 12.6085
  Epoch [10/200] Loss_D: 0.2340  Loss_G: 12.1860
  Epoch [11/200] Loss_D: 0.0175  Loss_G: 4.9454
  Epoch [12/200] Loss_D: 2.3437  Loss_G: 25.0381
  Epoch [13/200] Loss_D: 0.0293  Loss_G: 11.9700
  Epoch [14/200] Loss_D: 0.2419  Loss_G: 5.6051
  Epoch [

100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]



🚀 Training final classifier...


100%|██████████| 43/43 [00:12<00:00,  3.37it/s]


Epoch 1/15 — Loss: 0.6199


...

Epoch 13/15 — Loss: 0.3707


100%|██████████| 43/43 [00:09<00:00,  4.42it/s]


Epoch 14/15 — Loss: 0.3705


100%|██████████| 43/43 [00:09<00:00,  4.49it/s]


Epoch 15/15 — Loss: 0.3700

✅ Final Accuracy: 69.50%
💾 Model saved!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import os
import re
import shutil
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

# ====================== PATHS & CONFIG ======================
DATASET_ROOT   = '/content/drive/MyDrive/Data'
IMAGE_SRC_DIR  = '/content/drive/MyDrive/xray_dataset'
CROPS_ROOT     = '/content/drive/MyDrive/xray_crops'
CLEAN_ROOT     = '/content/drive/MyDrive/xray_clean'
SYNTHETIC_ROOT = '/content/drive/MyDrive/xray_synthetic'
COMBINED_ROOT  = '/content/drive/MyDrive/xray_combined'

CLASSES     = ['organic', 'inorganic', 'explosives']
NUM_CLASSES = len(CLASSES)
IMAGE_EXTS  = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')

LABEL_MAP = {
    'EXPLOSIVE': 'explosives',
    'ORGANIC':   'organic',
    'INORGANIC': 'inorganic',
    'METAL':     'inorganic',
}

IMAGE_SIZE  = 64
BATCH_SIZE  = 64
LATENT_DIM  = 100
EPOCHS_GAN  = 200
LR_G = LR_D = 0.0002

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# ====================== STEP 1: CROP EXPLOSIVES FROM XMLs ======================
print("\n📂 Indexing source images by UUID...")
uuid_to_path = {}
for f in os.listdir(IMAGE_SRC_DIR):
    if f.lower().endswith(IMAGE_EXTS):
        uuid_match = re.search(r'([a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12})', f)
        if uuid_match:
            uuid_to_path[uuid_match.group(1)] = os.path.join(IMAGE_SRC_DIR, f)
print(f"   Indexed {len(uuid_to_path)} images")

for cls in CLASSES:
    os.makedirs(os.path.join(CROPS_ROOT, cls), exist_ok=True)

explosives_dir = os.path.join(DATASET_ROOT, 'explosives')
xml_files      = [f for f in os.listdir(explosives_dir) if f.endswith('.xml')]
print(f"\n📦 Parsing {len(xml_files)} XML files and cropping bounding boxes...")

counts        = {cls: 0 for cls in CLASSES}
missing_uuids = []

for xml_file in tqdm(xml_files):
    xml_path = os.path.join(explosives_dir, xml_file)
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        uuid_match = re.search(
            r'([a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12})',
            xml_file
        )
        if not uuid_match:
            continue
        uuid = uuid_match.group(1)

        img_path = uuid_to_path.get(uuid)
        if img_path is None:
            missing_uuids.append(uuid)
            continue

        img          = Image.open(img_path).convert('RGB')
        img_w, img_h = img.size

        for obj in root.findall('object'):
            label_tag = obj.find('n')
            bndbox    = obj.find('bndbox')
            if label_tag is None or bndbox is None:
                continue

            cls_name = LABEL_MAP.get(label_tag.text.strip().upper())
            if cls_name is None:
                continue

            xmin = max(0,     int(bndbox.find('xmin').text))
            ymin = max(0,     int(bndbox.find('ymin').text))
            xmax = min(img_w, int(bndbox.find('xmax').text))
            ymax = min(img_h, int(bndbox.find('ymax').text))

            if xmax <= xmin or ymax <= ymin:
                continue

            crop      = img.crop((xmin, ymin, xmax, ymax))
            save_name = f"{uuid}_obj{counts[cls_name]:05d}.png"
            crop.save(os.path.join(CROPS_ROOT, cls_name, save_name))
            counts[cls_name] += 1

    except Exception as e:
        print(f"  ⚠️ {xml_file}: {e}")

print(f"\n✅ Cropping complete!")
for cls, n in counts.items():
    print(f"   {cls:12s}: {n} crops")
if missing_uuids:
    print(f"\n⚠️  {len(missing_uuids)} XMLs had no matching image in xray_dataset/")

# ====================== STEP 2: BUILD CLEAN DATASET ======================
# organic + inorganic: copy real images from Data/
# explosives: use crops from XMLs (no raw images exist)
print("\n🧹 Building clean dataset...")

for cls in CLASSES:
    dst = os.path.join(CLEAN_ROOT, cls)
    os.makedirs(dst, exist_ok=True)

    if cls in ('organic', 'inorganic'):
        # Copy real images, skip any XML files
        src = os.path.join(DATASET_ROOT, cls)
        count = 0
        for f in os.listdir(src):
            if f.lower().endswith(IMAGE_EXTS):
                shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
                count += 1
        print(f"   {cls:12s}: {count} real images copied")

    else:  # explosives
        # Copy XML-derived crops
        src = os.path.join(CROPS_ROOT, 'explosives')
        count = 0
        for f in os.listdir(src):
            if f.lower().endswith(IMAGE_EXTS):
                shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
                count += 1
        print(f"   {cls:12s}: {count} crops from XMLs")

        # Also add any organic/inorganic crops to their respective folders
for cls in ('organic', 'inorganic'):
    src   = os.path.join(CROPS_ROOT, cls)
    dst   = os.path.join(CLEAN_ROOT, cls)
    count = 0
    if os.path.exists(src):
        for f in os.listdir(src):
            if f.lower().endswith(IMAGE_EXTS):
                shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
                count += 1
    if count:
        print(f"   {cls:12s}: +{count} extra crops added from XML annotations")

print("✅ Clean dataset ready")

# ====================== DCGAN MODELS ======================
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x): return self.main(x)

class Discriminator(nn.Module):
    def __init__(self, ndf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x): return self.main(x)

def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# ====================== STEP 3: TRAIN GAN PER CLASS ======================
def train_gan_for_class(class_name, class_idx):
    print(f"\n🔥 Training GAN for {class_name}...")
    full_dataset  = datasets.ImageFolder(root=CLEAN_ROOT, transform=transform)
    class_indices = [i for i, t in enumerate(full_dataset.targets) if t == class_idx]

    if len(class_indices) == 0:
        raise RuntimeError(f"No images found for '{class_name}' in CLEAN_ROOT.")

    print(f"   {len(class_indices)} images found")
    dataloader = DataLoader(
        Subset(full_dataset, class_indices),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
    )

    netG = Generator().to(device);     netG.apply(weights_init)
    netD = Discriminator().to(device); netD.apply(weights_init)
    criterion  = nn.BCELoss()
    optimizerD = optim.Adam(netD.parameters(), lr=LR_D, betas=(0.5, 0.999))
    optimizerG = optim.Adam(netG.parameters(), lr=LR_G, betas=(0.5, 0.999))

    for epoch in range(EPOCHS_GAN):
        for i, (real_imgs, _) in enumerate(dataloader):
            real_imgs  = real_imgs.to(device)
            b_size     = real_imgs.size(0)
            label_real = torch.ones(b_size,  device=device)
            label_fake = torch.zeros(b_size, device=device)

            netD.zero_grad()
            loss_D_real = criterion(netD(real_imgs).view(-1), label_real)
            loss_D_real.backward()
            noise       = torch.randn(b_size, LATENT_DIM, 1, 1, device=device)
            fake_imgs   = netG(noise)
            loss_D_fake = criterion(netD(fake_imgs.detach()).view(-1), label_fake)
            loss_D_fake.backward()
            optimizerD.step()

            netG.zero_grad()
            loss_G = criterion(netD(fake_imgs).view(-1), label_real)
            loss_G.backward()
            optimizerG.step()

            if i % 50 == 0:
                print(f"  Epoch [{epoch+1}/{EPOCHS_GAN}] "
                      f"Loss_D: {loss_D_real.item()+loss_D_fake.item():.4f}  "
                      f"Loss_G: {loss_G.item():.4f}")

    os.makedirs(SYNTHETIC_ROOT, exist_ok=True)
    torch.save(netG.state_dict(), f"{SYNTHETIC_ROOT}/generator_{class_name}.pth")
    print(f"✅ GAN for {class_name} done!")
    return netG

def generate_synthetic(class_name, netG, num_images=400):
    save_dir = os.path.join(SYNTHETIC_ROOT, class_name)
    os.makedirs(save_dir, exist_ok=True)
    netG.eval()
    with torch.no_grad():
        for i in range(num_images):
            noise = torch.randn(1, LATENT_DIM, 1, 1, device=device)
            fake  = (netG(noise).cpu().squeeze(0) * 0.5 + 0.5).clamp(0, 1)
            transforms.ToPILImage()(fake).save(
                os.path.join(save_dir, f"syn_{i:04d}.png"))
    print(f"✅ Generated {num_images} synthetic {class_name} images")

for idx, cls in enumerate(CLASSES):
    netG = train_gan_for_class(cls, idx)
    generate_synthetic(cls, netG, num_images=400)

print("\n🎉 All GANs trained!")

# ====================== STEP 4: BUILD COMBINED DATASET ======================
print("\n🔗 Building combined dataset...")
if os.path.exists(COMBINED_ROOT):
    shutil.rmtree(COMBINED_ROOT)

for cls in CLASSES:
    dst = os.path.join(COMBINED_ROOT, cls)
    os.makedirs(dst, exist_ok=True)

    # Real / crop images
    for f in os.listdir(os.path.join(CLEAN_ROOT, cls)):
        if f.lower().endswith(IMAGE_EXTS):
            shutil.copy2(os.path.join(CLEAN_ROOT, cls, f), os.path.join(dst, f))

    # Synthetic images
    syn_dir = os.path.join(SYNTHETIC_ROOT, cls)
    for f in os.listdir(syn_dir):
        if f.lower().endswith(IMAGE_EXTS):
            shutil.copy(os.path.join(syn_dir, f), os.path.join(dst, f))

    count = len([f for f in os.listdir(dst) if f.lower().endswith(IMAGE_EXTS)])
    print(f"   {cls:12s}: {count} total images (real + synthetic)")

print("✅ Combined dataset ready")

# ====================== STEP 5: TRAIN FINAL CLASSIFIER ======================
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

combined_dataset = datasets.ImageFolder(root=COMBINED_ROOT, transform=transform)
train_size = int(0.8 * len(combined_dataset))
val_size   = len(combined_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(combined_dataset, [train_size, val_size])
train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=0)

print("\n🚀 Training final classifier...")
for epoch in range(15):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}/15 — Loss: {running_loss/len(train_loader):.4f}")

model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        _, predicted   = torch.max(model(images), 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n✅ Final Accuracy: {100 * correct / total:.2f}%")
torch.save(model.state_dict(), '/content/drive/MyDrive/xray_classifier_3class.pth')
print("💾 Model saved to Drive!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda

📂 Indexing source images by UUID...
   Indexed 669 images

📦 Parsing 120 XML files and cropping bounding boxes...


100%|██████████| 120/120 [00:00<00:00, 261.65it/s]



✅ Cropping complete!
   organic     : 0 crops
   inorganic   : 0 crops
   explosives  : 0 crops

⚠️  120 XMLs had no matching image in xray_dataset/

🧹 Building clean dataset...
   organic     : 451 real images copied
